# Lab 04.1 — Apache Ozone come storage per Spark

**Ozone** è un object store distribuito del mondo Hadoop, pensato per miliardi di oggetti (dove HDFS soffre per i metadati in memoria sul NameNode).

| Componente | Ruolo |
|---|---|
| **Ozone Manager (OM)** | namespace: volumi, bucket, chiavi |
| **Storage Container Manager (SCM)** | blocchi, *container* e repliche sui DataNode |
| **DataNode** | memorizzano i dati |
| **S3 Gateway** | espone i bucket con l'API Amazon S3 |

Gerarchia dei dati: **volume → bucket → chiave** (es. `/lab/vendite/2024/file.parquet`).

Spark accede a Ozone con il filesystem Hadoop **`ofs://`** (*Rooted Ozone FileSystem*): `ofs://<om>/<volume>/<bucket>/<chiave>`.
Il client (`ozone-filesystem-hadoop3.jar`) è già incluso nell'immagine Spark.

UI: Ozone Manager http://localhost:9874 · SCM http://localhost:9876

In [ ]:
import os
from pyspark.sql import SparkSession, functions as F

spark = (SparkSession.builder
         .appName("Lab04-Ozone")
         .master(os.environ.get("SPARK_MASTER", "local[*]"))
         # Implementazione del filesystem ofs:// e indirizzo dell'Ozone Manager
         .config("spark.hadoop.fs.ofs.impl", "org.apache.hadoop.fs.ozone.RootedOzoneFileSystem")
         .config("spark.hadoop.ozone.om.address", "om:9862")
         # Un solo DataNode nel laboratorio → una sola replica
         .config("spark.hadoop.ozone.replication", "1")
         .getOrCreate())
sc = spark.sparkContext
OZ = "ofs://om"
print("Master:", sc.master)

## 1. Volumi e bucket
Con `ofs://` il primo livello del percorso è il **volume**, il secondo il **bucket**: si creano come directory.

In [ ]:
jvm = sc._jvm
Path = jvm.org.apache.hadoop.fs.Path
fs = jvm.org.apache.hadoop.fs.FileSystem.get(jvm.java.net.URI(f"{OZ}/"), sc._jsc.hadoopConfiguration())

fs.mkdirs(Path(f"{OZ}/lab/vendite"))      # volume "lab", bucket "vendite"
fs.mkdirs(Path(f"{OZ}/lab/testi"))        # bucket "testi"

def ls(percorso):
    for st in fs.listStatus(Path(percorso)):
        tipo = "d" if st.isDirectory() else "-"
        print(f"{tipo} {st.getLen():>12,d}  {st.getPath()}")

print("Volumi:");             ls(f"{OZ}/")
print("\nBucket del volume lab:"); ls(f"{OZ}/lab")

## 2. Caricare un file e fare Word Count
`copyFromLocalFile` scrive una chiave nel bucket; Spark poi la legge in parallelo come un normale file Hadoop.

In [ ]:
fs.copyFromLocalFile(Path("file:///home/jovyan/datasets/alice_in_wonderland.txt"), Path(f"{OZ}/lab/testi/alice.txt"))
ls(f"{OZ}/lab/testi")

import re
top = (sc.textFile(f"{OZ}/lab/testi/alice.txt")
         .flatMap(lambda l: re.findall(r"[a-z]+", l.lower()))
         .filter(lambda w: len(w) > 3)
         .map(lambda w: (w, 1))
         .reduceByKey(lambda a, b: a + b)
         .takeOrdered(8, key=lambda kv: -kv[1]))
print(top)

## 3. DataFrame e Parquet partizionato su Ozone

In [ ]:
vendite = (spark.range(0, 500_000)
           .withColumn("anno", (2023 + F.col("id") % 3).cast("int"))
           .withColumn("prodotto", F.concat(F.lit("P"), (F.col("id") % 50).cast("string")))
           .withColumn("importo", F.round(F.rand(11) * 200, 2)))

vendite.write.partitionBy("anno").mode("overwrite").parquet(f"{OZ}/lab/vendite/parquet")
ls(f"{OZ}/lab/vendite/parquet")

In [ ]:
spark.read.parquet(f"{OZ}/lab/vendite/parquet").createOrReplaceTempView("vendite")
spark.sql("""
    SELECT anno, COUNT(*) AS n, CAST(SUM(importo) AS DECIMAL(14, 2)) AS totale
    FROM vendite
    WHERE anno >= 2024
    GROUP BY anno ORDER BY anno
""").show()

## 4. Altri formati sullo stesso bucket

In [ ]:
top_prodotti = spark.sql("""
    SELECT prodotto, ROUND(SUM(importo), 2) AS totale
    FROM vendite GROUP BY prodotto ORDER BY totale DESC LIMIT 10
""")
top_prodotti.coalesce(1).write.mode("overwrite").option("header", True).csv(f"{OZ}/lab/vendite/report_csv")
top_prodotti.coalesce(1).write.mode("overwrite").json(f"{OZ}/lab/vendite/report_json")
ls(f"{OZ}/lab/vendite")
spark.read.json(f"{OZ}/lab/vendite/report_json").show(3)

## 5. Rinomina atomica
I bucket creati tramite `ofs://` hanno layout **FILE_SYSTEM_OPTIMIZED**: le directory esistono davvero e `rename` è un'operazione sui metadati
(non una copia, come accade negli object store puri). È ciò che rende sicuri i *commit* dei job Spark.

In [ ]:
fs.delete(Path(f"{OZ}/lab/vendite/report_finale_csv"), True)      # se il notebook è rieseguito
print("rename:", fs.rename(Path(f"{OZ}/lab/vendite/report_csv"), Path(f"{OZ}/lab/vendite/report_finale_csv")))
ls(f"{OZ}/lab/vendite")

In [ ]:
spark.stop()